To train this agent, click _Runtime_ and press _Run all_. Make sure you've enabled a free Tesla T4 GPU!

<div class="align-center">
<a href="https://github.com/openpipe/art"><img src="https://github.com/openpipe/art/raw/main/assets/ART_pill.png" height="50"></a>
<a href="https://discord.gg/zbBHRUpwf4"><img src="https://github.com/openpipe/art/raw/main/assets/Discord_pill.png" height="50"></a>
<a href="https://openpipe.ai/blog/art-e-mail-agent"><img src="https://github.com/openpipe/art/raw/main/assets/ART_E_pill.png" height="50"></a>

Questions? Join the Discord and ask away! For feature requests or to leave a star, visit our [Github](https://github.com/openpipe/art).

</div>

<a href="https://art.openpipe.ai/"><img src="https://github.com/openpipe/art/raw/main/assets/Header_separator.png" height="5"></a>

This notebook shows how to train a Qwen 2.5 3B model to play tic tac toe. It will demonstrate how to set up a multi-turn agent, how to train it, and how to evaluate it.

Completions will be logged to OpenPipe, and metrics will be logged to Weights & Biases.

You will learn how to construct an [agentic environment](#Environment), how to define a [rollout](#Rollout), and how to run a [training loop](#Loop).


### WARNING:

If you are running in Google Colab and installing numpy does not say "Requirement already satisfied: numpy<2.0.0" then click "Runtime" and "Restart Session."


In [1]:
# make sure we're using numpy 1.*.*
import numpy as np

if (np.__version__).startswith("1."):
    print("Numpy version is 1.*.*, you're good to go!")
else:
    raise ValueError("Please restart your runtime using the above instructions!")

Numpy version is 1.*.*, you're good to go!


### Environment Variables

Later on in the notebook, we'll be creating a model that can automatically logs metrics to Weights & Biases. In order to do so, you'll need to provide your Weights & Biases API key as an environment variable.

You can also optionally initiate an OpenPipe client to report completions to a [dashboard](https://app.openpipe.ai) to get a feel for what the completions your model is generating look like, and how they change over time. Logging to OpenPipe is free, but is not required for training!


In [2]:
import os


# Optional
WANDB_API_KEY = "308e1be7938bf7a7c366afc0522fb9fc0d8cf1ad"
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

# Optional
OPENPIPE_API_KEY = "opk_90bbdee21c12388777592e16fe2b570014e64c7f18"
OPENPIPE_API_KEY = ""
if OPENPIPE_API_KEY:
    os.environ["OPENPIPE_API_KEY"] = OPENPIPE_API_KEY

### Creating a Model

Now that we've defined the rules of our environment, we can create a model that will learn to play 2048. We'll use a Qwen 2.5 3B model for this example. The `name` parameter will be associated with a wandb run, and the `base_model` parameter is the model that we'll be training a LoRA on top of.

In [3]:
import art
from dotenv import load_dotenv
import random

from openpipe.client import OpenPipe
from art.local import LocalBackend

load_dotenv()

op_client = OpenPipe()
print("OpenPipe client initialized")

random.seed(42)

backend = LocalBackend(path="./.art")

INFO 06-28 19:45:54 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-28 19:45:54 [__init__.py:239] Automatically detected platform cuda.
OpenPipe client initialized


### Creating a Model

Now that we've defined the rules of our environment, we can create a model that will learn to play tic tac toe. We'll use a Qwen 2.5 3B model for this example. The `name` parameter will be associated with a wandb run, and the `base_model` parameter is the model that we'll be training a LoRA on top of.


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# model_name = "Qwen/Qwen2.5-3B-Instruct"
model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit"

project_name = "easy2hard-dac_agent"
run_name = "003"

model = art.TrainableModel(
    name=run_name, project=project_name, base_model=model_name,
)
await model.register(backend)

/home/guycoh/AgentDaC/.venv/lib/python3.11/site-packages/art/__init__.py:11: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth  # type: ignore


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 06-28 19:46:22 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-28 19:46:22 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.5.1: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Llama-3.3-70B-Instruct-bnb-4bit with actual GPU utilization = 64.56%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.15 GB.
Unsloth: Using conservativeness = 1.0.

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:02<00:17,  2.56s/it]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:05<00:15,  2.58s/it]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:08<00:13,  2.70s/it]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:09<00:09,  2.40s/it]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:11<00:06,  2.24s/it]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:14<00:04,  2.43s/it]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:17<00:02,  2.44s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:19<00:00,  2.50s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:19<00:00,  2.47s/it]

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:02<00:14,  2.10s

INFO 06-28 19:47:36 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 06-28 19:47:37 [model_runner.py:1140] Model loading took 37.0489 GiB and 38.354897 seconds
INFO 06-28 19:47:57 [worker.py:287] Memory profiling takes 18.51 seconds
INFO 06-28 19:47:57 [worker.py:287] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.65) = 51.10GiB
INFO 06-28 19:47:57 [worker.py:287] model weights take 37.05GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 2.51GiB; the rest of the memory reserved for KV Cache is 11.46GiB.
INFO 06-28 19:47:57 [executor_base.py:112] # cuda blocks: 2346, # CPU blocks: 1228
INFO 06-28 19:47:57 [executor_base.py:117] Maximum concurrency for 10000 tokens per request: 3.75x
INFO 06-28 19:48:09 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI

Capturing CUDA graph shapes: 100%|██████████| 27/27 [01:21<00:00,  3.02s/it]


INFO 06-28 19:49:33 [model_runner.py:1592] Graph capturing finished in 84 secs, took 8.71 GiB
INFO 06-28 19:49:33 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 116.17 seconds
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'k_norm', 'q_norm', 'pre_feedforward_layernorm']
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'k_norm', 'q_norm', 'pre_feedforward_layernorm']


Unsloth 2025.5.1 patched 80 layers with 80 QKV layers, 80 O layers and 80 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


### Defining a Rollout

<a name="Rollout"></a>

A rollout is a single episode of an agent performing its task. It generates one or more trajectories, which are lists of messages and choices.

In this example, the rollout function generates a game of tic tac toe, and the agent plays it until the game is finished. It then returns a trajectory which contains all the `system` and `user` messages presented to the agent, as well as all the `choices` that the agent made.

When the game is finished the `reward` for the agent's performance is calculated based on whether the agent won, lost, drew, or errored, which is then assigned to the trajectory.

This rollout function will be called many times in parallel during each step of the training loop.

In [5]:
import re

def extract_text_between_markers(
    text: str, start_marker: str, end_marker: str
) -> list[str]:
    """
    Extracts all instances of text between two specific markers in a string.

    Args:
        text: The input string to parse.
        start_marker: The beginning marker.
        end_marker: The ending marker.

    Returns:
        A list of strings, where each string is an instance of text found between the markers.
    """
    # Create a regex pattern to find text non-greedily between the markers
    # re.escape is used to escape any special characters in the markers
    # (.*?) matches any character (except newline) zero or more times, non-greedily
    pattern = re.escape(start_marker) + r"(.*?)" + re.escape(end_marker)

    # Find all non-overlapping matches of the pattern in the string
    matches = re.findall(pattern, text, re.DOTALL)

    return matches

In [ ]:
import art
import openai
import time
import math
from pydantic import BaseModel
from openai.types.chat import (
    ChatCompletionDeveloperMessageParam,
    ChatCompletionToolMessageParam,
)
import os
from openai import AsyncOpenAI
from typing import defaultdict
from art.openai import patch_openai
from sys_prompt import SystemPrompt, DaCSystemPrompt
from dac_agent import DACAgent

timings = dict(
    total_chat_time=[],
    total_rollout_time=[],
)


@art.retry(exceptions=(openai.LengthFinishReasonError,))
async def rollout(
    model_inference_name,
    client: AsyncOpenAI,
    question: str,
    answer: str,
    
) -> art.Trajectory:
    total_chat_time = 0
    total_rollout_time = time.time()

    agent = DACAgent(
        client=client,
        model=model_inference_name,
        # model_system_message=SystemPrompt.Qwen,
        dac_sys_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3,
        leaf_sys_prompt=DaCSystemPrompt.dac_sys_prompt_v2_3_leaf,
        # dac_sys_prompt=prompt,
        max_depth=1,
        max_length=3,  # Limit the number of messages in a single chat
    )

    prompt = "Please answer the following question, write the final answer in \\boxed\{\}."
    message = {
        "role": "user",
        "content": f"{prompt} \"{question}\"",
    }
    try:
        requested_at = time.time()
        trajectory = await agent.chat(message)
        total_chat_time += time.time() - requested_at
    except openai.LengthFinishReasonError as e:
        raise e
    except Exception as e:
        print("caught exception generating chat completion")
        print(e)
        global failing_trajectory
        failing_trajectory = trajectory
        raise e

    content = trajectory.messages()[-1]["content"]
    agent_answer = extract_text_between_markers(content, "boxed{", "}")
    if len(agent_answer) == 0:
        trajectory.reward -= 1
    else:
        agent_answer = agent_answer[-1].strip()  # Get the last answer
        if agent_answer == answer:
            trajectory.reward += 1
            trajectory.metrics["correct_answer"] = 1
        else:
            trajectory.metrics["correct_answer"] = 0
    # Add the answer and agent_answer to the trajectory metrics
    trajectory.metadata["answer"] = answer
    trajectory.metadata["agent_answer"] = agent_answer

    total_rollout_time = time.time() - total_rollout_time
    timings["total_chat_time"].append(total_chat_time)
    timings["total_rollout_time"].append(total_rollout_time)

    return trajectory

<a name="Loop"></a>

### Training Loop

The training loop is where the magic happens. For each of the 100 steps defined below, the rollout function will be called 200 times in parallel. This means that 200 games will be played at once. Each game will produce a trajectory, which will be used to update the model.

The `gather` step will wait for all of the trajectories to be generated, then it will delete all but the most recent checkpoint and train the model on the new trajectories.

Inference will be blocked until the training is complete.


In [7]:
# use vllm inference client
# vllm_port = 8200
# vllm_client = AsyncOpenAI(base_url=f"http://localhost:{vllm_port}/v1", api_key="default")
# vllm_client = patch_openai(vllm_client)


# use the art  client
art_port = 8001
model.inference_base_url = model.inference_base_url.replace(":8000", f":{art_port}")
# client = model.openai_client()

## Training - ART + VLLM

In [8]:
from vllm_client import VLLMClient
vllm_ports = [8200, 8201]  # List of VLLM ports to connect to
vllm_clients:list[VLLMClient] = []
for vllm_port in vllm_ports:
    vllm_client = VLLMClient(port=vllm_port, base_model=model_name)
    vllm_clients.append(vllm_client)

inference_clients = [vllm_client.client for vllm_client in vllm_clients] + [model.openai_client()]  # Add the art client to the list
n_clients = len(vllm_clients) + 1  # Add 1 for the art client

In [9]:
from datasets import load_dataset, Dataset
import random

easy2hard_dataset = load_dataset("furonghuang-lab/Easy2Hard-Bench", "E2H-AMC")

# The dataset is usually split into 'train' and 'test'
train_data:Dataset = easy2hard_dataset['train']
test_data:Dataset = easy2hard_dataset['eval']

In [ ]:
from art.utils.output_dirs import (
    get_output_dir_from_model_properties,
    get_step_checkpoint_dir,
)

epochs = 20
n_rollouts = 5
n_groups = 1



prev_step_checkpoint_dir = None
step_checkpoint_dir = None
inference_name = model_name

for i in range(await model.get_step(), epochs):
    print(f"Starting step {i} for model {model.name}")
    train_data = train_data.shuffle(seed=42)
    epoch_data = train_data.take(n_rollouts)  # Take the first n_rollouts samples for training
    epoch_data = epoch_data.sort("item_difficulty", reverse=False)

    output_dir = get_output_dir_from_model_properties(
        project_name, name=run_name, art_path="./art"
    )
    prev_step_checkpoint_dir = step_checkpoint_dir
    step_checkpoint_dir = get_step_checkpoint_dir(output_dir, i) if i > 0 else None
    step_checkpoint_dir = (step_checkpoint_dir.replace("./art/", ".art/") if step_checkpoint_dir is not None else None)

    if prev_step_checkpoint_dir is not None:
        print(f"Unloading lora")
        [client.unload_lora(run_name) for client in vllm_clients]
    if step_checkpoint_dir is not None:
        print(f"Loading lora from {step_checkpoint_dir}")
        results = [client.load_lora(run_name, step_checkpoint_dir) for client in vllm_clients]
        if all(result["success"] for result in results):
            inference_name = run_name
        else:
            print(f"Failed to load lora from {step_checkpoint_dir}, using base model {model_name}")
            break

    # inference_clients = [{"client":vllm_client.client, "name":vllm_client.get_inference_name()} for vllm_client in vllm_clients] + [{"client":model.openai_client(), "name":model.get_inference_name()}]  # Add the art client to the list
    inference_clients = [{"client":model.openai_client(), "name":model.get_inference_name()}]  # Add the art client to the list
    train_groups = await art.gather_trajectory_groups(
        (
            art.TrajectoryGroup(
                rollout(
                    inference_clients[i % n_clients]['name'],
                    inference_clients[i % n_clients]['client'],
                    question=sample['problem'][0],
                    answer=sample['answer'][0].strip('{}'), 
                ) for i, sample in enumerate(epoch_data.iter(batch_size=1)) # Number of rollouts per group
            ) for _ in range(n_groups) # Number of groups to gather
        ),
        pbar_desc="gather",
    )

    await model.delete_checkpoints()
    await model.train(train_groups, config=art.TrainConfig(learning_rate=2e-5))

Starting step 0 for model 003


gather:   0%|          | 0/5 [00:00<?, ?it/s]

Packed 5 trajectories into 3 sequences of length 2048


train:   0%|          | 0/3 [00:00<?, ?it/s]

RuntimeError: Size does not match at dimension 1 expected index [1, 1024, 1] to be no larger than self [1, 784, 128256] apart from dimension 2

In [ ]:
mean_chat_time = np.mean(timings["total_chat_time"])
mean_rollout_time = np.mean(timings["total_rollout_time"])
print(f"Mean chat time: {mean_chat_time:.2f} seconds")
print(f"Mean rollout time: {mean_rollout_time:.2f} seconds")
print(f"Total chat time: {sum(timings['total_chat_time']):.2f} seconds")
print(f"Total rollout time: {sum(timings['total_rollout_time']):.2f} seconds")


<div class="align-center">
<a href="https://github.com/openpipe/art"><img src="https://github.com/openpipe/art/raw/notebooks/assets/ART_pill.png" height="50"></a>
<a href="https://discord.gg/zbBHRUpwf4"><img src="https://github.com/openpipe/art/raw/notebooks/assets/Discord_pill.png" height="50"></a>
<a href="https://openpipe.ai/blog/art-e-mail-agent"><img src="https://github.com/openpipe/art/raw/main/assets/ART_E_pill.png" height="50"></a>

Questions? Join the Discord and ask away! For feature requests or to leave a star, visit our [Github](https://github.com/openpipe/art).

</div>
